In [41]:
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

# Access variables using os.getenv()
# **Required environment variables:**
# - `URI` → Neo4j database URI
# - `USERNAME` / `PASSWORD` → Neo4j authentication
# - `OPENAI_KEY` → OpenAI API key for LLM-based querying
database_url = os.getenv("URI")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")
OPENAI_KEY = os.getenv("OPENAI_KEY")

## Getting the Data(Pre fetching)

In this section, we define and collect the research papers that will serve as
the knowledge source for our graph. These papers focus on agentic workflows,
attention mechanisms, and reinforcement learning–based optimization.


In [42]:
## list of the urls for the arxiv papers-
## list down all the urls you think are relevant
## or let a agent search out all the relevant paper links for you 
## Source Papers (ArXiv)

# Below is a curated list of ArXiv paper URLs relevant to:
# - Agentic workflows
# - Attention mechanisms
# - Policy optimization for LLM agents

# These are **abstract pages**, which allows flexibility if we later want to:
# - Scrape metadata
# - Validate relevance
# - Convert to PDF programmatically

raw_arxiv_links = [
  "https://arxiv.org/abs/2503.12434", 
  "https://arxiv.org/abs/2503.21460", 
  "https://arxiv.org/abs/2402.01680", 
  "https://arxiv.org/abs/2402.16823",
  "https://arxiv.org/abs/2502.04492",
  "https://arxiv.org/abs/2506.02718",
  "https://arxiv.org/abs/2512.16848",
  "https://arxiv.org/abs/2507.21504",
  "https://arxiv.org/abs/2506.09171",
  "https://arxiv.org/abs/2508.19598",
  "https://arxiv.org/abs/2510.14548",
  "https://arxiv.org/abs/2511.14460",
  "https://arxiv.org/abs/2505.11821",
]


## Paper Processing & Knowledge Extraction Strategy

To efficiently extract structured knowledge from long research papers, we
follow a multi-stage pipeline:

1. **Content Pruning**
   - Remove References and Appendices
   - These sections often account for ~30% of tokens and add little semantic value
     for knowledge graph construction

2. **Recursive Chunking**
   - Split papers into chunks of 2,000–4,000 tokens
   - Use ~10% overlap to preserve cross-chunk relationships

3. **Parallel Structured Extraction**
   - Each chunk is processed independently using an LLM
   - Outputs conform to a predefined Pydantic schema

4. **Graph Merging (Neo4j MERGE)**
   - Nodes and relationships are merged, not duplicated
   - This ensures:
     - Entity deduplication across papers
     - Incremental graph growth


In [ ]:
## Downloading Paper PDFs

# ArXiv PDF URLs follow a predictable format based on the paper ID.
# Here, we:
# 1. Extract the base ArXiv ID from each abstract URL
# 2. Convert it into a direct PDF link
# 3. Download all PDFs locally for downstream processing

final_list = []
prepend_url = 'https://arxiv.org/pdf/'

for i in raw_arxiv_links:
    final_list.append(prepend_url + i[-10:]+ '.pdf')    ## # Extracts the ArXiv paper ID (e.g., 2503.12434)


## download the files and do the processing or you can write a script to do the processing on fly
for i in final_list:
    !wget -P /files {i}


## Cost Optimization: Local LLMs

To reduce LLM API costs during large-scale extraction, this pipeline can also
run on **local models** via Ollama.

Quantized models such as **LLaMA 3B** are often sufficient for:
- Named Entity Recognition (NER)
- Relationship extraction
- Schema-constrained outputs

> This is especially useful during iterative development or large batch runs.


In [ ]:
## we can use the Pydantic Schema for structured data extraction
## Structured Knowledge Extraction Schema

# We use a Pydantic schema to enforce **consistent, machine-readable outputs**
# from the LLM.

# This schema ensures:
# - Deterministic node and relationship formats
# - Easy validation before graph insertion
# - Compatibility with Neo4j and Cypher queries


# import ollama
from pydantic import BaseModel, Field
from typing import List

# 1. Define our Schema (The Map)
class Node(BaseModel):
    id: str = Field(description="Unique entity name (e.g., 'Attention Mechanism')")
    label: str = Field(description="Type: Technology, Method, Metric, or Paper")

class Relationship(BaseModel):
    source: str = Field(description="Origin node ID")
    target: str = Field(description="Destination node ID")
    type: str = Field(description="Relationship type (e.g., 'IMPROVES', 'USES', 'AUTHORED_BY')")

class KnowledgeGraph(BaseModel):
    nodes: List[Node]
    relationships: List[Relationship]

In [ ]:
## I have the extracted data already with me
## Extracted Knowledge Graph Data

# Below is a **pre-extracted** representation of the knowledge graph obtained
# from the processed papers.

# This data follows the previously defined schema and includes:
# - High-level concepts
# - Methods and technologies
# - Relationships capturing lineage and influence


data = {
    "nodes": [
        # HIGH-LEVEL CONCEPTS
        {"id": "Agentic Workflows", "label": "Concept", "properties": {"description": "Autonomous loops for LLM task execution"}},
        {"id": "Attention Mechanism", "label": "Technology", "properties": {"origin": "2017", "evolution": "Standard to Sparse/Linear"}},
        {"id": "Policy Optimization", "label": "Method", "properties": {"category": "Reinforcement Learning"}},
        {"id": "Synthetic Data Generation", "label": "Method", "properties": {"use_case": "Self-Correction"}},
        {"id": "ARPO", "label": "Method", "properties": {"full_name": "Agentic Reinforced Policy Optimization", "arxiv_id": "2507.19849", "year": 2025}},
        {"id": "CCPO", "label": "Method", "properties": {"full_name": "Conformal Constrained Policy Optimization", "arxiv_id": "2511.11828", "year": 2025}},
        {"id": "Deep Search", "label": "Concept", "properties": {"focus": "Long-horizon reasoning", "year": 2025}},
        {"id": "Contrastive Reasoning", "label": "Method", "properties": {"focus": "Tool Usage", "year": 2024}},
        {"id": "Tree Search RL", "label": "Method", "properties": {"arxiv_id": "2509.21240", "year": 2025}},
        {"id": "Step-wise RL", "label": "Method", "properties": {"focus": "Granular Rewards", "year": 2024}},
        {"id": "Instruction-Policy Co-Evolution", "label": "Method", "properties": {"year": 2025}},
        {"id": "Trustworthy Optimization", "label": "Concept", "properties": {"focus": "Verifiable Data"}},
        {"id": "Linear Attention", "label": "Technology", "properties": {"improvement": "Efficiency for long context"}},
        {"id": "Tool Usage", "label": "Task", "properties": {}},
        {"id": "Multi-Turn Dialogue", "label": "Task", "properties": {}},
        {"id": "Self-Correction", "label": "Task", "properties": {}},
        {"id": "Flash-Attention-3", "label": "Method", "properties": {
                "full_name": "FlashAttention-3: Fast and I/O-Aware Attention", 
                "year": 2024,
                "arxiv_id": "2507.08608",
                "impact": "Crucial for long-context agentic reasoning"
            }
        },
        {
            "id": "Agentic-Transformer", 
            "label": "Method", 
            "properties": {
                "full_name": "Agentic Transformer Architectures", 
                "year": 2024, 
                "arxiv_id": "2305.16554",
                "focus": "Adaptive attention for multi-step tasks"
            }
        },
        {
            "id": "Sparse-Agent-Attention", 
            "label": "Method", 
            "properties": {
                "full_name": "Sparse Attention for Long-Horizon Agents", 
                "year": 2024, 
                "arxiv_id": "2210.17540"
            }
        }
    ],
    "relationships": [
        # The 'Lineage' (This answers your complex queries)
        {"source": "Agentic Workflows", "target": "Attention Mechanism", "type": "BUILT_UPON"},
        {"source": "Linear Attention", "target": "Attention Mechanism", "type": "IMPROVES"},
        {"source": "ARPO", "target": "Agentic Workflows", "type": "OPTIMIZES"},
        {"source": "ARPO", "target": "Policy Optimization", "type": "IMPLEMENTS"},
        {"source": "CCPO", "target": "Agentic Workflows", "type": "CONSTRAINS"},
        {"source": "Contrastive Reasoning", "target": "Agentic Workflows", "type": "ENHANCES_REASONING"},
        {"source": "Contrastive Reasoning", "target": "Tool Usage", "type": "OPTIMIZES"},
        {"source": "Tree Search RL", "target": "Deep Search", "type": "ENABLES"},
        {"source": "Step-wise RL", "target": "Multi-Turn Dialogue", "type": "IMPROVES"},
        {"source": "Instruction-Policy Co-Evolution", "target": "Policy Optimization", "type": "EVOLVED_FROM"},
        {"source": "Verifiable Synthetic Data", "target": "Trustworthy Optimization", "type": "ENSURES"},
        {"source": "Flash-Attention-3", "target": "Attention Mechanism", "type": "IMPROVES"},
        {"source": "Flash-Attention-3", "target": "Agentic Workflows", "type": "OPTIMIZES"},
        {"source": "Agentic-Transformer", "target": "Attention Mechanism", "type": "IMPROVES"},
        {"source": "Agentic-Transformer", "target": "Agentic Workflows", "type": "BUILT_UPON"},
        {"source": "Sparse-Agent-Attention", "target": "Attention Mechanism", "type": "IMPROVES"},
        {"source": "Sparse-Agent-Attention", "target": "Agentic Workflows", "type": "USES"}
    ]
}

## Ingesting Data into Neo4j

This function pushes nodes and relationships into Neo4j using `MERGE`
instead of `CREATE`.

Why `MERGE`?
- Prevents duplicate entities
- Allows incremental graph updates
- Enables multi-paper knowledge fusion


In [ ]:

from neo4j import GraphDatabase
def push_to_neo4j(data):
    driver = GraphDatabase.driver(uri= database_url, auth=(USERNAME,PASSWORD))
    with driver.session() as session:
        # Push Nodes
        for node in data["nodes"]:
            session.run("""
                MERGE (n:Entity {id: $id})
                SET n.label = $label, n += $props
            """, id=node["id"], label=node["label"], props=node["properties"])
            
        # Push Relationships
        for rel in data["relationships"]:
            session.run("""
                MATCH (a:Entity {id: $source})
                MATCH (b:Entity {id: $target})
                MERGE (a)-[r:RELATIONSHIP {type: $type}]->(b)
            """, source=rel["source"], target=rel["target"], type=rel["type"])
    driver.close()

push_to_neo4j(data)

## Natural Language Graph Querying

In this section, we enable **natural language querying** over the knowledge graph
using LangChain + Neo4j.

The LLM translates user questions into Cypher queries while respecting:
- Graph schema
- Relationship constraints
- Flexible text matching


In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 1. Connect to your database
graph = Neo4jGraph(url = database_url, username = USERNAME, password = PASSWORD)

# 2. Initialize the LLM (GPT-4o is highly recommended for Cypher)
llm = ChatOpenAI(model="gpt-4o",
                 temperature=0.2,
                 api_key = OPENAI_KEY)

#3 initialize the prompt template 
CYPHER_GENERATION_TEMPLATE = """Task: Generate Cypher statement to query a graph database.
Instructions:
1. Use ONLY the provided relationship types and properties in the schema.
2. Use 'id' property for matching names.
3. CRITICAL: Use `toLower(n.id) CONTAINS toLower('search_term')` for flexibility.
4. If a year is mentioned, apply it to the 'Method' or 'Paper' node, not the concept.

Schema:
{schema}

Example:
Question: "Find papers that improved Attention Mechanism for Agentic Workflows"
Cypher Query:
MATCH (att:Entity) WHERE toLower(att.id) CONTAINS 'attention'
MATCH (aw:Entity) WHERE toLower(aw.id) CONTAINS 'agentic'
MATCH (p:Entity)-[*1..2]-(att)
MATCH (p)-[*1..2]-(aw)
RETURN DISTINCT p.id, p.year

Question: {question}
Cypher Query:"""

## added few shot prompting

CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], 
    template=CYPHER_GENERATION_TEMPLATE
)

# 3. Create the Chain
chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    verbose=True, 
    cypher_prompt=CYPHER_PROMPT,
    allow_dangerous_requests=True   # Required in newer versions
)

## Example Query

We now query the graph to identify papers from 2024 that improved
attention mechanisms in agentic workflows.

This demonstrates:
- Temporal filtering
- Multi-hop graph reasoning
- Practical research discovery


In [39]:
# 4. Ask the raw query
result = chain.invoke({"query": "Find all papers that improved the Attention Mechanism for Agentic Workflows in 2024."})
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Entity) WHERE p.year = 2024
MATCH (p)-[:IMPROVES]->(att:Entity) WHERE toLower(att.id) CONTAINS 'attention'
MATCH (p)-[:OPTIMIZES|BUILT_UPON|USES]->(aw:Entity) WHERE toLower(aw.id) CONTAINS 'agentic'
RETURN p.id, p.full_name

Full Context:
[{'p.id': 'Flash-Attention-3', 'p.full_name': 'FlashAttention-3: Fast and I/O-Aware Attention'}, {'p.id': 'Flash-Attention-3', 'p.full_name': 'FlashAttention-3: Fast and I/O-Aware Attention'}, {'p.id': 'Flash-Attention-3', 'p.full_name': 'FlashAttention-3: Fast and I/O-Aware Attention'}, {'p.id': 'Flash-Attention-3', 'p.full_name': 'FlashAttention-3: Fast and I/O-Aware Attention'}, {'p.id': 'Agentic-Transformer', 'p.full_name': 'Agentic Transformer Architectures'}, {'p.id': 'Sparse-Agent-Attention', 'p.full_name': 'Sparse Attention for Long-Horizon Agents'}, {'p.id': 'Sparse-Agent-Attention', 'p.full_name': 'Sparse Attention for Long-Horizon Agents'}, {'p.id': 'Sparse-Agent